# Text.

In [1]:
import os
import pandas as pd 
from sklearn.preprocessing import LabelEncoder
from tools import load_sets_paths
import csv

In [2]:
all_possible_keys = [
    "N",
    "X",
    "C",
    "C#",
    "D",
    "D#",
    "E",
    "F",
    "F#",
    "G",
    "G#",
    "A",
    "A#",
    "B"
]

all_possible_qualities = [
    "None",
    "min",
    "maj",
    "7"
]

tones_normalized_equiv = {
    # Flat notes
    "Cb": "B",
    "Db": "C#",
    "Eb": "D#",
    "Fb": "E",
    "Gb": "F#",
    "Ab": "G#",
    "Bb": "A#"
    # Maybe Sharp notes too for future data updates
}

# Helper functions
def equalize_tone(tone):
    if tone in tones_normalized_equiv:
        return tones_normalized_equiv[tone]
    else:
        return tone
        

In [3]:
# Fun variables
lab_type = "majmin.lab"
dataset_size = 300

# Load data paths
data_path = "McGill-Billboard"
output_path = "output"

data_paths = load_sets_paths(data_path)

In [4]:
def process_singles():
    # Label encoder for turning keys into into integers (sorted alphabetically)
    key_encoder = LabelEncoder()
    key_encoder.fit(all_possible_keys)

    # Same for qualities
    quality_encoder = LabelEncoder()
    quality_encoder.fit(all_possible_qualities)

    # Descriptive comment
    dataset_index = 0
    for data_path in data_paths:
        dataset_index += 1
        if dataset_index > dataset_size:
            break

        print(f"Processing for {data_path}")

        # Load chroma stuff
        chroma_cols = ['filepath', 'timestamp'] + [f'chroma_{i}' for i in range(24)]
        df_chroma = pd.read_csv(os.path.join(data_path, "bothchroma.csv"), header=None, names=chroma_cols)
        df_chroma = df_chroma.drop(columns=['filepath'])

        # Load lab stuff
        df_labels = pd.read_csv(os.path.join(data_path, lab_type), sep='\t', header=None, names=['start_time', 'end_time', 'chord'])

        # Merge the two dataframes, ordered, structured, neat
        df_merged = pd.merge_asof(df_chroma, 
            df_labels, 
            left_on='timestamp', 
            right_on='start_time',
            direction='backward')

        # uuhh
        df_aligned = df_merged[df_merged['timestamp'] < df_merged['end_time']].copy()

        # Create key and quality columns with default values (in case a file doesn't contain ":")
        df_aligned[["key", "quality"]] = ["X", "None"]
        if df_aligned["chord"].str.contains(":", na=False).any():
            df_aligned[["key", "quality"]] = df_aligned["chord"].str.split(":", n=1, expand=True)
        else:
            df_aligned[["key"]] = df_aligned[["chord"]]

        # Drop the time columns from the labels file, they are no longer needed
        df_aligned = df_aligned.drop(columns=['start_time', 'end_time', 'chord'])

        # Equalize tones
        df_aligned["key"] = df_aligned["key"].apply(equalize_tone)

        # Encode
        df_aligned["key"] = key_encoder.transform(df_aligned["key"])
        df_aligned["quality"] = quality_encoder.transform(df_aligned["quality"])
        #print(df_aligned.head(50))

        print("Saving to its own file!")

In [5]:
#process_singles()

In [6]:
def process_monolith():
    # Label encoder for turning keys into into integers (sorted alphabetically)
    key_encoder = LabelEncoder()
    key_encoder.fit(all_possible_keys)

    # Same for qualities
    quality_encoder = LabelEncoder()
    quality_encoder.fit(all_possible_qualities)

    # Descriptive comment
    df_monolith = pd.DataFrame()
    dataset_index = 0
    for data_path in data_paths:
        dataset_index += 1
        if dataset_index > dataset_size:
            break

        print(f"Processing for {data_path}")

        # Load chroma stuff
        chroma_cols = ['filepath', 'timestamp'] + [f'chroma_{i}' for i in range(24)]
        df_chroma = pd.read_csv(os.path.join(data_path, "bothchroma.csv"), header=None, names=chroma_cols)
        df_chroma = df_chroma.drop(columns=['filepath'])

        # Load lab stuff
        df_labels = pd.read_csv(os.path.join(data_path, lab_type), sep='\t', header=None, names=['start_time', 'end_time', 'chord'])

        # Merge the two dataframes, ordered, structured, neat
        df_merged = pd.merge_asof(df_chroma, 
            df_labels, 
            left_on='timestamp', 
            right_on='start_time',
            direction='backward')

        # uuhh
        df_aligned = df_merged[df_merged['timestamp'] < df_merged['end_time']].copy()

        # Create key and quality columns with default values (in case a file doesn't contain ":")
        df_aligned[["key", "quality"]] = ["X", "None"]
        if df_aligned["chord"].str.contains(":", na=False).any():
            df_aligned[["key", "quality"]] = df_aligned["chord"].str.split(":", n=1, expand=True)
        else:
            df_aligned[["key"]] = df_aligned[["chord"]]

        # Drop the time columns from the labels file, they are no longer needed
        df_aligned = df_aligned.drop(columns=['start_time', 'end_time', 'chord'])

        # Equalize tones
        df_aligned["key"] = df_aligned["key"].apply(equalize_tone)

        # Encode
        df_aligned["key"] = key_encoder.transform(df_aligned["key"])
        df_aligned["quality"] = quality_encoder.transform(df_aligned["quality"])
        #print(df_aligned.head(50))

        print("Appending to monolith!")
        #df_collection.append(df_aligned)
        df_monolith = pd.concat([df_monolith, df_aligned], ignore_index=True)
    
    # Write whole thing to NEW file
    print(df_monolith.shape)
    df_monolith.to_csv(os.path.join(output_path, "bigass_file.csv"), mode="w+", index=False)


In [7]:
process_monolith()

Processing for McGill-Billboard\lmao.txt


FileNotFoundError: [Errno 2] No such file or directory: 'McGill-Billboard\\lmao.txt\\bothchroma.csv'